# 02 · Set one temperature target

Apply the configured target once. Readback verifies the reported target, not thermal settling. Inspect equipment after an uncertain write; never automatically replay it.

Run top to bottom in a **Python 3.12+** kernel. This notebook includes all configuration and setup; no sibling example files are imported. Run only one hardware notebook/process at a time.

**Prepare the filled cooling loop before connecting.** T255P connection sends U, takes serial control and can change AUTO START to RUN, even with target writes disabled. Software disconnect does not stop the pump/chiller or restore its target. Physical hardware has not been validated with this package.

## 1. Environment and imports

From the repository root, run `uv sync --locked --group notebooks`, then `uv run --locked --group notebooks jupyter lab`. Select the environment kernel described in examples/README.md. The script-only dashboard (07_dashboard.py) needs `--extra gui`. Restart the kernel after installation or driver source changes. Imports open no serial ports.

In [1]:
import sys
from pathlib import Path

# Prefer checkout sources when launched from the repository or examples folder.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src/tark_chiller/controller.py").is_file():
        sys.path.insert(0, str(candidate / "src"))
        break

In [2]:
from tark_chiller.controller import TarkChiller
from tark_chiller.protocols import T255P, ProtocolError

## 2. Choose the port

This example connects to a **ThermoTek T255P**. Set `PORT` to its serial port (for example `"COM9"`). Adapter metadata identifies the USB adapter, not the attached chiller.

The connection cell below spells out the T255P profile: 9600 baud, 8N1, XON/XOFF, no hardware flow control, and RTS/DTR deasserted. Check that your adapter/cable supports these line levels. Distilled-water target bounds are 5–45 °C, in 0.1 °C steps. See the [hardware guide](../docs/hardware.md) for wiring and the separate CAL protocol.

In [3]:
PORT = "COM9"  # Set the identified serial port, e.g. "COM9".

TARGET_C = 17.0  # Choose an approved target within your setup limits.

## 3. Connection settings

The next operation cell constructs the driver directly with the settings shown. Construction opens nothing. Its context connects and releases the port, including on a read error or interrupt. Among the notebooks, only example 02 enables target writes; dashboard script 07 also enables explicit GUI target requests.

## 4. Set one temperature target

This lesson reads H0, sends one M with `set_setpoint()`, then reads H0 again with `get_memory()`. The setter checks only its acknowledgement. The following readback and comparison are explicit notebook steps. A failed acknowledgement or readback must never trigger an automatic resend.


In [4]:
if not isinstance(PORT, str) or not PORT.strip():
    raise ValueError('Hardware not configured: set PORT, e.g. "COM9". No port was opened.')

chiller = TarkChiller(
    {
        "port": PORT,
        "baudrate": 9600,
        "bytesize": 8,
        "parity": "N",
        "stopbits": 1,
        "xonxoff": True,
        "rtscts": False,
        "dsrdtr": False,
        "rts": False,
        "dtr": False,
    },
    protocol=T255P(allow_writes=True),
    setpoint_range=(5.0, 45.0),
    coolant_source="ThermoTek T255P Rev C pp9/20; verify installed distilled water",
)

with chiller:
    before = chiller.get_memory()  # H0 only; context entry established the session with U.
    print(f"Existing setpoint: {before['setpoint_c']:.2f} C")
    print(f"Applying {TARGET_C:g} C")

    # M only: validate this one acknowledgement; never retry an uncertain write.
    chiller.set_setpoint(TARGET_C)
    print("Target acknowledgement received.")

    # 3. Read the target again. This is a target check, not a settling test.
    # H0 is a separate command. A failure here does not authorize resending M.
    try:
        reported_c = chiller.get_memory()["setpoint_c"]
    except BaseException as error:
        error.add_note("Target acknowledged; readback failed. Inspect the panel; do not replay M.")
        raise
    print(f"Reported setpoint: {reported_c:.2f} C")
    if reported_c != TARGET_C:
        raise ProtocolError(
            "Target readback differs from request; inspect the panel, do not replay"
        )
    print("Target readback verified. Observe the actual liquid temperature separately.")

# 4. The connection is closed; the equipment keeps its current target/state.
# No automatic restoration, pump control or equipment stop is sent.

Existing setpoint: 17.00 C
Applying 17 C
Target acknowledgement received.
Reported setpoint: 17.00 C
Target readback verified. Observe the actual liquid temperature separately.


## 5. Check results and finish

Observe the actual liquid temperature separately. Matching target readback does not establish settling.

Context cleanup closes software resources, including after a read error or kernel interrupt. If cleanup fails, resources may remain owned: allow pending I/O to finish and retry `chiller.disconnect(timeout_s=5.0)` if `chiller` was created. Do not rerun configuration over a live connection. A forced kernel termination cannot guarantee cleanup. Follow the equipment's physical shutdown procedure separately.

Restart the kernel after updating the driver, then rerun from the top. Target changes are allowed in standby and with reported alarms. `collect_status()` reports structured mode and alarm fields; the serial protocol has no alarm-clear command. Resolve the physical cause and follow the panel procedure for clearing it.